<a href="https://colab.research.google.com/github/vanessapeterka/a-sigla/blob/main/Vanessa_ex02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas numpy requests


In [ ]:
import requests
import pandas as pd
import numpy as np
import json
import time
from datetime import datetime, timezone
from IPython.display import display

In [ ]:
# Identificação da estação
boia_id = 12
estacao = "RJ-4"

# Período de aquisição
data_inicio = "2025-10-20"
data_fim = "2026-11-03"

# Conversão para UTC
inicio = pd.Timestamp(data_inicio, tz="UTC")

# O limite superior é exclusivo:
# assim incluímos todo o dia 30/11/2026
fim = pd.Timestamp(data_fim, tz="UTC") + pd.Timedelta(days=1)

print("Estação:", estacao)
print("Início:", inicio)
print("Fim:", fim)

Estação: RJ-4
Início: 2025-10-20 00:00:00+00:00
Fim: 2026-11-04 00:00:00+00:00


In [ ]:
url_estacao = (
    "https://simcosta.furg.br/q/stations/byName/rj4"
)

resposta = requests.get(url_estacao, timeout=60)
resposta.raise_for_status()

metadados = resposta.json()

print(json.dumps(metadados, indent=4, ensure_ascii=False))

{
    "id": 12,
    "name": "rj4",
    "name_display": "RJ-4",
    "installation": "2017-08-28T13:01:00.000Z",
    "city": "Rio de Janeiro",
    "state": "RJ",
    "responsible": "UFRJ",
    "header": "/L1 data \n/missing = NULL\n\n/Acknowledging or Crediting SiMCosta as Information Source : (Product or data name) courtesy of the Brazilian Coastal Monitoring System - SiMCosta  (www.simcosta.furg.br), Universidade Federal do Rio Grande - FURG.\n/Magnetic Declination Reference: NOAA NCEI Geomagnetic Modeling Team; British Geological Survey: World Magnetic Model. NOAA National Centers for Environmental Information. https://www.ncei.noaa.gov/products/world-magnetic-model\n/Disclaimer: All products published on this website are not intended to be used for navigational or operational purposes.\n\n/Anemometer measuring height: 3.0m",
    "adcp_header": "/sensor_mount = Buoy\n/ADCP_orientation = Downward\n\n/ADCP config:\n/Starting measuring depth (m) = 0.71\n/Number of Cells = 20\n/Cell Size 

In [ ]:
url_api = "https://simcosta.furg.br/api/intrans_data"

# Criar intervalos mensais
intervalos = pd.date_range(
    start=inicio,
    end=fim,
    freq="MS"
)

# Garantir que o início do período também seja considerado
limites = [inicio]

for data in intervalos:
    if inicio < data < fim:
        limites.append(data)

limites.append(fim)

limites = sorted(set(limites))

lista_dados = []

for i in range(len(limites) - 1):

    t1 = int(limites[i].timestamp())
    t2 = int(limites[i+1].timestamp())

    parametros = {
        "boiaID": boia_id,
        "type": "json",
        "time1": t1,
        "time2": t2,
        "params": "perfil_correntes"
    }

    print(
        f"Consultando: {limites[i]} até {limites[i+1]}"
    )

    resposta = requests.get(
        url_api,
        params=parametros,
        timeout=120
    )

    resposta.raise_for_status()

    dados = resposta.json()

    if isinstance(dados, list):
        registros = dados

    elif isinstance(dados, dict):
        registros = dados.get("data", dados)

        if isinstance(registros, dict):
            registros = [registros]

    else:
        raise ValueError("Formato inesperado retornado pela API.")

    if registros:
        lista_dados.extend(registros)

    time.sleep(0.5)

print("Total de registros recebidos:", len(lista_dados))

Consultando: 2025-10-20 00:00:00+00:00 até 2025-11-01 00:00:00+00:00
Consultando: 2025-11-01 00:00:00+00:00 até 2025-12-01 00:00:00+00:00
Consultando: 2025-12-01 00:00:00+00:00 até 2026-01-01 00:00:00+00:00
Consultando: 2026-01-01 00:00:00+00:00 até 2026-02-01 00:00:00+00:00
Consultando: 2026-02-01 00:00:00+00:00 até 2026-03-01 00:00:00+00:00
Consultando: 2026-03-01 00:00:00+00:00 até 2026-04-01 00:00:00+00:00
Consultando: 2026-04-01 00:00:00+00:00 até 2026-05-01 00:00:00+00:00
Consultando: 2026-05-01 00:00:00+00:00 até 2026-06-01 00:00:00+00:00
Consultando: 2026-06-01 00:00:00+00:00 até 2026-07-01 00:00:00+00:00
Consultando: 2026-07-01 00:00:00+00:00 até 2026-08-01 00:00:00+00:00
Consultando: 2026-08-01 00:00:00+00:00 até 2026-09-01 00:00:00+00:00
Consultando: 2026-09-01 00:00:00+00:00 até 2026-10-01 00:00:00+00:00
Consultando: 2026-10-01 00:00:00+00:00 até 2026-11-01 00:00:00+00:00
Consultando: 2026-11-01 00:00:00+00:00 até 2026-11-04 00:00:00+00:00
Total de registros recebidos: 1428

In [ ]:
df = pd.json_normalize(lista_dados)

# Verificar se existem registros
if df.empty:
    raise ValueError(
        "A API não retornou registros para o período solicitado."
    )

# Construir a variável temporal
if "timestamp" in df.columns:

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        utc=True
    )

elif all(c in df.columns for c in
         ["YEAR", "MONTH", "DAY", "HOUR", "MINUTE", "SECOND"]):

    df["timestamp"] = pd.to_datetime(
        df[["YEAR", "MONTH", "DAY",
            "HOUR", "MINUTE", "SECOND"]]
    ).dt.tz_localize("UTC")

else:
    raise ValueError("Não foi encontrada a informação temporal.")

# Garantir o período solicitado
df = df[
    (df["timestamp"] >= inicio) &
    (df["timestamp"] < fim)
].copy()

# Organizar cronologicamente
df = df.sort_values("timestamp")

# Eliminar duplicatas temporais, se houver
df = df.drop_duplicates(
    subset=["timestamp"],
    keep="first"
)

df = df.reset_index(drop=True)

print("Dimensões:", df.shape)
print("Primeiro registro:", df["timestamp"].min())
print("Último registro:", df["timestamp"].max())

display(df.head())

Dimensões: (14284, 47)
Primeiro registro: 2025-10-20 00:21:40+00:00
Último registro: 2026-11-03 23:21:40+00:00


,timestamp,YEAR,MONTH,DAY,HOUR,MINUTE,SECOND,Avg_Cell(001)_dir_n,Avg_Cell(002)_dir_n,Avg_Cell(003)_dir_n,...,Avg_Cell(011)_mag,Avg_Cell(012)_mag,Avg_Cell(013)_mag,Avg_Cell(014)_mag,Avg_Cell(015)_mag,Avg_Cell(016)_mag,Avg_Cell(017)_mag,Avg_Cell(018)_mag,Avg_Cell(019)_mag,Avg_Cell(020)_mag
0,2025-10-20 00:21:40+00:00,2025,10,20,0,21,40,281.008,301.008,314.008,...,174,149,130,120,107,105,89,89,97,128
1,2025-10-20 00:51:40+00:00,2025,10,20,0,51,40,303.008,313.008,321.008,...,217,202,210,216,168,127,77,59,41,40
2,2025-10-20 01:21:40+00:00,2025,10,20,1,21,40,306.008,308.008,311.008,...,233,225,204,201,139,132,100,40,34,58
3,2025-10-20 01:51:40+00:00,2025,10,20,1,51,40,318.008,330.008,336.008,...,294,265,245,215,186,124,84,72,57,81
4,2025-10-20 02:21:40+00:00,2025,10,20,2,21,40,297.008,328.008,326.008,...,306,266,239,229,184,136,85,78,76,87


In [ ]:
arquivo_csv = "RJ4_SiMCosta_correntes_2025_2026.csv"

df.to_csv(
    arquivo_csv,
    index=False,
    sep=";",
    decimal=",",
    encoding="utf-8-sig"
)

print("Arquivo salvo:", arquivo_csv)

Arquivo salvo: RJ4_SiMCosta_correntes_2025_2026.csv


In [ ]:
from google.colab import files

files.download(arquivo_csv)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Quantidade de dados
print("Número de registros:", len(df))
print("Número de variáveis:", len(df.columns))

# Intervalo temporal
print("\nIntervalo temporal:")
print(df["timestamp"].min(), "até", df["timestamp"].max())

# Intervalos entre medições
print("\nIntervalos entre registros:")
print(
    df["timestamp"]
    .diff()
    .value_counts()
    .head(10)
)

# Percentual de dados ausentes
ausentes = (
    df.isna().mean() * 100
).sort_values(ascending=False)

display(ausentes.head(30))

# Identificar colunas de corrente
colunas_corrente = [
    c for c in df.columns
    if "_mag" in c or "_dir_n" in c
]

print("\nVariáveis de corrente encontradas:")
print(colunas_corrente)

Número de registros: 14284
Número de variáveis: 47

Intervalo temporal:
2025-10-20 00:21:40+00:00 até 2026-11-03 23:21:40+00:00

Intervalos entre registros:
timestamp
0 days 00:30:00    13127
0 days 01:00:00      834
0 days 01:30:00      154
0 days 02:00:00       48
0 days 02:30:00       20
0 days 03:00:00       16
0 days 03:30:00       11
0 days 04:00:00        5
0 days 00:25:05        3
0 days 00:30:02        3
Name: count, dtype: int64


,0
timestamp,0.0
YEAR,0.0
MONTH,0.0
DAY,0.0
HOUR,0.0
MINUTE,0.0
SECOND,0.0
Avg_Cell(001)_dir_n,0.0
Avg_Cell(002)_dir_n,0.0
Avg_Cell(003)_dir_n,0.0



Variáveis de corrente encontradas:
['Avg_Cell(001)_dir_n', 'Avg_Cell(002)_dir_n', 'Avg_Cell(003)_dir_n', 'Avg_Cell(004)_dir_n', 'Avg_Cell(005)_dir_n', 'Avg_Cell(006)_dir_n', 'Avg_Cell(007)_dir_n', 'Avg_Cell(008)_dir_n', 'Avg_Cell(009)_dir_n', 'Avg_Cell(010)_dir_n', 'Avg_Cell(011)_dir_n', 'Avg_Cell(012)_dir_n', 'Avg_Cell(013)_dir_n', 'Avg_Cell(014)_dir_n', 'Avg_Cell(015)_dir_n', 'Avg_Cell(016)_dir_n', 'Avg_Cell(017)_dir_n', 'Avg_Cell(018)_dir_n', 'Avg_Cell(019)_dir_n', 'Avg_Cell(020)_dir_n', 'Avg_Cell(001)_mag', 'Avg_Cell(002)_mag', 'Avg_Cell(003)_mag', 'Avg_Cell(004)_mag', 'Avg_Cell(005)_mag', 'Avg_Cell(006)_mag', 'Avg_Cell(007)_mag', 'Avg_Cell(008)_mag', 'Avg_Cell(009)_mag', 'Avg_Cell(010)_mag', 'Avg_Cell(011)_mag', 'Avg_Cell(012)_mag', 'Avg_Cell(013)_mag', 'Avg_Cell(014)_mag', 'Avg_Cell(015)_mag', 'Avg_Cell(016)_mag', 'Avg_Cell(017)_mag', 'Avg_Cell(018)_mag', 'Avg_Cell(019)_mag', 'Avg_Cell(020)_mag']
